# SPPACEU — 2

Run from the repository directory after installing `requirements.txt`. Configuration uses environment variables described in README.md. External processing is disabled by default. Choose one video-screening variant (steps 3–7); do not run all variants against the same outputs.


In [ ]:
import os
from pathlib import Path
from notebook_runtime import DATA_DIR, RECORDS_DIR, MODEL_NAME, load_openai_key, prepare_run

RUN_ANALYSIS = False  # Enable explicitly after reviewing inputs and API/download settings.
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
#!/usr/bin/env python3

import json
import time
from pathlib import Path
from typing import Optional, Dict, Any, List

import pandas as pd
from openai import OpenAI

In [ ]:
# =========================
# CONFIGURATION
# =========================
INPUT_XLSX = DATA_DIR / "youtube_data.xlsx"
OUTPUT_XLSX = DATA_DIR / "playlist_ai_analysis.xlsx"
LOG_XLSX = DATA_DIR / "playlist_ai_processing_log.xlsx"
DIAGNOSTICS_XLSX = DATA_DIR / "playlist_items_diagnostics.xlsx"

API_KEY_PATH = None  # Read OPENAI_API_KEY from the environment.

SHEET_PLAYLISTS = "playlists"
SHEET_PODCASTS = "podcasts"

ITEM_SHEETS = [
    "playlist_video_membership",
    "playlists_items",
    "podcasts_items",
]

MODEL_NAME = os.environ.get("OPENAI_MODEL", "").strip()

MAX_RETRIES = 3

REQUEST_TIMEOUT = 180

SLEEP_BETWEEN_REQUESTS = 1.0

MAX_COMPLETION_TOKENS = 4000

SKIP_EXISTING = True

# Для теста можно поставить 10. Для полного анализа — None.

LIMIT_ROWS: Optional[int] = None

# Сколько названий видео из списка отправлять модели.

MAX_TITLES_PER_PLAYLIST = 100

# Сколько описаний видео отправлять модели.

MAX_DESCRIPTIONS_PER_PLAYLIST = 25

# Максимальная длина одного описания видео.

MAX_DESCRIPTION_CHARS = 600

In [ ]:
# =========================
# REQUIRED COLUMNS
# =========================
REQUIRED_LIST_COLUMNS = [
    "playlist_id",
    "channel_title",
    "playlist_title",
]

REQUIRED_ITEM_COLUMNS = [
    "playlist_id",
    "video_id",
    "video_title",
]

OPTIONAL_LIST_COLUMNS = [
    "channel",
    "channel_stem",
    "channel_id",
    "playlist_url",
    "description",
    "published_at",
    "privacy_status",
    "podcast_status",
    "item_count",
    "thumbnail_url",
    "source_section",
]

OPTIONAL_ITEM_COLUMNS = [
    "channel",
    "channel_stem",
    "channel_id",
    "channel_title",
    "source_section",
    "playlist_title",
    "playlist_url",
    "playlist_item_id",
    "playlist_position",
    "playlist_item_published_at",
    "video_description",
    "video_url",
    "video_owner_channel_id",
    "video_owner_channel_title",
    "video_published_at",
    "privacy_status",
    "thumbnail_url",
]


# =========================
# SYSTEM PROMPT
# =========================
SYSTEM_INSTRUCTION = """
You are analyzing YouTube playlist metadata for a sociological research project.

Research topic:
"Social practices of public audit and corruption exposure in Ukraine."

Your task is to classify a YouTube playlist based only on the provided metadata:
- channel title;
- playlist title;
- playlist description, if available;
- item count, if available;
- video titles included in the playlist;
- selected video descriptions included in the playlist;
- publication range and basic structural indicators, if available.

Do not invent facts. Do not use external knowledge about the channel or playlist.
Classify only what is supported by the metadata.

Return ONLY valid JSON. No markdown. No explanatory text outside JSON.

Use this JSON schema:

{
  "playlist_topic": "string",
  "secondary_topics": ["string"],
  "playlist_function": "string",
  "playlist_type": "string",
  "corruption_relevance_category": "string",
  "public_audit_relevance_category": "string",
  "ukraine_relevance_category": "string",
  "playlist_relevance_score": 0.0,
  "classification_confidence": 0.0,
  "inclusion_status": "string",
  "manual_review_required": true,
  "evidence_markers": ["string"],
  "risk_notes": ["string"],
  "short_rationale": "string"
}

Definitions:

playlist_topic:
Main thematic field of the playlist.

secondary_topics:
Additional thematic fields if visible.

playlist_function:
One of:
- investigative
- analytical
- news
- civic_monitoring
- public_accountability
- anti_corruption_education
- governmental_or_institutional_information
- legal_or_judicial_information
- political_commentary
- civic_activism
- educational
- archive
- promotional
- entertainment
- mixed
- unclear

playlist_type:
One of:
- thematic_series
- news_or_updates_collection
- investigative_collection
- civic_monitoring_collection
- anti_corruption_collection
- public_audit_collection
- educational_collection
- institutional_collection
- archive_collection
- podcast_collection
- technical_or_auto_generated
- mixed_or_unclear

corruption_relevance_category:
One of:
- direct_corruption_focus
- indirect_governance_or_accountability_focus
- possible_contextual_relevance
- low_relevance
- unclear

public_audit_relevance_category:
One of:
- direct_public_audit_focus
- civic_monitoring_or_watchdog_focus
- transparency_or_accountability_focus
- possible_contextual_relevance
- low_relevance
- unclear

ukraine_relevance_category:
One of:
- direct_ukraine_focus
- ukrainian_language_or_context
- possible_ukraine_context
- no_visible_ukraine_focus
- unclear

playlist_relevance_score:
A number from 0.00 to 1.00 estimating relevance to the research topic.
Use:
0.80-1.00 = highly relevant
0.60-0.79 = relevant but requires review
0.40-0.59 = partially/contextually relevant
0.00-0.39 = low relevance

classification_confidence:
A number from 0.00 to 1.00 estimating confidence based on available metadata.

inclusion_status:
One of:
- include_core
- include_after_manual_review
- contextual_or_borderline
- exclude
- insufficient_data

manual_review_required:
true if metadata is ambiguous, incomplete, contradictory, or relevance is borderline.

evidence_markers:
Short words or phrases from the playlist title, playlist description, video titles, or video descriptions that support the classification.

risk_notes:
Possible limitations, such as empty description, too few video titles, generic playlist title, mixed topic, technical playlist, missing Ukrainian context, etc.

short_rationale:
2-4 sentences explaining the classification in academic language.
""".strip()


# =========================
# HELPERS
# =========================
def load_api_key() -> str:
    return load_openai_key(API_KEY_PATH)


def safe_str(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def validate_columns(df: pd.DataFrame, required: List[str], sheet_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in sheet '{sheet_name}': {missing}")


def sheet_exists(xls: pd.ExcelFile, sheet_name: str) -> bool:
    return sheet_name in xls.sheet_names


def read_sheet_if_exists(xls: pd.ExcelFile, sheet_name: str) -> pd.DataFrame:
    if not sheet_exists(xls, sheet_name):
        print(f"WARNING: Sheet not found: {sheet_name}")
        return pd.DataFrame()
    return pd.read_excel(xls, sheet_name=sheet_name)


def normalize_playlist_id(df: pd.DataFrame) -> pd.DataFrame:
    if "playlist_id" in df.columns:
        df["playlist_id"] = df["playlist_id"].astype(str).str.strip()
    return df


def add_list_source(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["list_source_sheet"] = source_name
    return df


def add_item_source(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["item_source_sheet"] = source_name
    return df


def load_lists(xls: pd.ExcelFile) -> pd.DataFrame:
    frames = []

    playlists = read_sheet_if_exists(xls, SHEET_PLAYLISTS)
    if not playlists.empty:
        validate_columns(playlists, REQUIRED_LIST_COLUMNS, SHEET_PLAYLISTS)
        playlists = normalize_playlist_id(playlists)
        playlists = add_list_source(playlists, SHEET_PLAYLISTS)
        frames.append(playlists)

    podcasts = read_sheet_if_exists(xls, SHEET_PODCASTS)
    if not podcasts.empty:
        validate_columns(podcasts, REQUIRED_LIST_COLUMNS, SHEET_PODCASTS)
        podcasts = normalize_playlist_id(podcasts)
        podcasts = add_list_source(podcasts, SHEET_PODCASTS)
        frames.append(podcasts)

    if not frames:
        raise ValueError("No playlist-like sheets found or loaded.")

    lists = pd.concat(frames, ignore_index=True)

    # Убираем полные дубли по playlist_id, если один и тот же список попал в два листа.
    lists = lists.drop_duplicates(subset=["playlist_id"], keep="first").copy()

    return lists


def load_items(xls: pd.ExcelFile) -> pd.DataFrame:
    frames = []

    for sheet in ITEM_SHEETS:
        df = read_sheet_if_exists(xls, sheet)
        if df.empty:
            continue

        validate_columns(df, REQUIRED_ITEM_COLUMNS, sheet)
        df = normalize_playlist_id(df)
        df = add_item_source(df, sheet)
        frames.append(df)

    if not frames:
        print("WARNING: No playlist item sheets found. Proceeding with playlist metadata only.")
        return pd.DataFrame()

    items = pd.concat(frames, ignore_index=True)

    # Убираем дубли, потому что playlist_video_membership может пересекаться с playlists_items/podcasts_items.
    subset_cols = [c for c in ["playlist_id", "video_id", "playlist_item_id"] if c in items.columns]
    if subset_cols:
        items = items.drop_duplicates(subset=subset_cols, keep="first").copy()
    else:
        items = items.drop_duplicates().copy()

    return items


def truncate_text(text: str, max_chars: int) -> str:
    text = safe_str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."


def safe_datetime_min_max(series: pd.Series) -> tuple[str, str]:

    dates = pd.to_datetime(series, errors="coerce", utc=True).dropna()

    if dates.empty:

        return "", ""

    return dates.min().isoformat(), dates.max().isoformat()

def aggregate_items(items: pd.DataFrame) -> pd.DataFrame:

    if items.empty:

        return pd.DataFrame(columns=[

            "playlist_id",

            "actual_video_count_from_items",

            "sample_video_titles",

            "sample_video_descriptions",

            "all_video_titles_count",

            "min_video_published_at",

            "max_video_published_at",

            "item_source_sheets",

        ])

    rows = []

    for playlist_id, group in items.groupby("playlist_id", dropna=False):

        group = group.copy()

        # ---------- SAFE SORTING ----------

        if "playlist_position" in group.columns:

            group["_sort_pos"] = pd.to_numeric(group["playlist_position"], errors="coerce")

            group = group.sort_values("_sort_pos", ascending=True, na_position="last")

        elif "video_published_at" in group.columns:

            group["_video_published_at_dt"] = pd.to_datetime(

                group["video_published_at"],

                errors="coerce",

                utc=True

            )

            group = group.sort_values("_video_published_at_dt", ascending=False, na_position="last")

        # ---------- VIDEO TITLES ----------

        titles = (

            group["video_title"]

            .dropna()

            .astype(str)

            .str.strip()

            .replace("", pd.NA)

            .dropna()

            .tolist()

        )

        # ---------- VIDEO DESCRIPTIONS ----------

        if "video_description" in group.columns:

            desc_values = (

                group["video_description"]

                .dropna()

                .astype(str)

                .str.strip()

                .replace("", pd.NA)

                .dropna()

                .tolist()

            )

        else:

            desc_values = []

        descriptions = [

            truncate_text(x, MAX_DESCRIPTION_CHARS)

            for x in desc_values[:MAX_DESCRIPTIONS_PER_PLAYLIST]

        ]

        # ---------- SAFE DATE RANGE ----------

        if "video_published_at" in group.columns:

            min_pub, max_pub = safe_datetime_min_max(group["video_published_at"])

        else:

            min_pub, max_pub = "", ""

        # ---------- ITEM SOURCES ----------

        if "item_source_sheet" in group.columns:

            item_sources = " | ".join(

                sorted(set(group["item_source_sheet"].dropna().astype(str)))

            )

        else:

            item_sources = ""

        rows.append({

            "playlist_id": safe_str(playlist_id),

            "actual_video_count_from_items": int(len(group)),

            "sample_video_titles": "\n".join(titles[:MAX_TITLES_PER_PLAYLIST]),

            "sample_video_descriptions": "\n\n---\n\n".join(descriptions),

            "all_video_titles_count": int(len(titles)),

            "min_video_published_at": min_pub,

            "max_video_published_at": max_pub,

            "item_source_sheets": item_sources,

        })

    return pd.DataFrame(rows)


def prepare_data() -> pd.DataFrame:
    if not INPUT_XLSX.exists():
        raise FileNotFoundError(f"Input Excel file not found: {INPUT_XLSX}")

    xls = pd.ExcelFile(INPUT_XLSX)

    lists = load_lists(xls)
    items = load_items(xls)
    agg = aggregate_items(items)

    merged = lists.merge(
        agg,
        on="playlist_id",
        how="left",
        validate="one_to_one"
    )

    fill_zero_cols = [
        "actual_video_count_from_items",
        "all_video_titles_count",
    ]
    for col in fill_zero_cols:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0).astype(int)

    fill_empty_cols = [
        "sample_video_titles",
        "sample_video_descriptions",
        "min_video_published_at",
        "max_video_published_at",
        "item_source_sheets",
    ]
    for col in fill_empty_cols:
        if col in merged.columns:
            merged[col] = merged[col].fillna("")

    # Diagnostics
    list_ids = set(lists["playlist_id"].dropna().astype(str))
    item_ids = set(items["playlist_id"].dropna().astype(str)) if not items.empty else set()

    diagnostics = pd.DataFrame([
        {"metric": "list_rows_after_dedup", "value": len(lists)},
        {"metric": "item_rows_after_dedup", "value": len(items)},
        {"metric": "unique_playlist_ids_in_lists", "value": len(list_ids)},
        {"metric": "unique_playlist_ids_in_items", "value": len(item_ids)},
        {"metric": "playlist_ids_with_items", "value": len(list_ids & item_ids)},
        {"metric": "playlist_ids_without_items", "value": len(list_ids - item_ids)},
        {"metric": "item_playlist_ids_not_in_lists", "value": len(item_ids - list_ids)},
        {"metric": "matched_share_of_lists", "value": len(list_ids & item_ids) / len(list_ids) if list_ids else 0.0},
    ])

    with pd.ExcelWriter(DIAGNOSTICS_XLSX, engine="openpyxl") as writer:
        diagnostics.to_excel(writer, index=False, sheet_name="summary")
        lists.head(500).to_excel(writer, index=False, sheet_name="lists_sample")
        items.head(500).to_excel(writer, index=False, sheet_name="items_sample")
        merged.head(500).to_excel(writer, index=False, sheet_name="merged_sample")

        if list_ids - item_ids:
            pd.DataFrame({"playlist_id": sorted(list_ids - item_ids)}).head(1000).to_excel(
                writer, index=False, sheet_name="lists_without_items"
            )

        if item_ids - list_ids:
            pd.DataFrame({"playlist_id": sorted(item_ids - list_ids)}).head(1000).to_excel(
                writer, index=False, sheet_name="items_without_lists"
            )

    print(f"Diagnostics saved to: {DIAGNOSTICS_XLSX}")
    print(f"Lists loaded: {len(lists)}")
    print(f"Items loaded: {len(items)}")
    print(f"Playlists with items: {len(list_ids & item_ids)} / {len(list_ids)}")

    if LIMIT_ROWS is not None:
        merged = merged.head(LIMIT_ROWS).copy()

    return merged


def make_row_key(row: pd.Series) -> str:
    playlist_id = safe_str(row.get("playlist_id", ""))
    if playlist_id:
        return playlist_id

    channel_title = safe_str(row.get("channel_title", ""))
    playlist_title = safe_str(row.get("playlist_title", ""))
    list_source = safe_str(row.get("list_source_sheet", ""))
    return f"{list_source}::{channel_title}::{playlist_title}"


def build_user_prompt(row: pd.Series) -> str:
    data = {
        "list_source_sheet": safe_str(row.get("list_source_sheet", "")),
        "playlist_id": safe_str(row.get("playlist_id", "")),
        "channel_id": safe_str(row.get("channel_id", "")),
        "channel_title": safe_str(row.get("channel_title", "")),
        "playlist_title": safe_str(row.get("playlist_title", "")),
        "playlist_url": safe_str(row.get("playlist_url", "")),
        "playlist_description": safe_str(row.get("description", "")),
        "declared_item_count": safe_str(row.get("item_count", "")),
        "actual_video_count_from_items": safe_str(row.get("actual_video_count_from_items", "")),
        "all_video_titles_count": safe_str(row.get("all_video_titles_count", "")),
        "min_video_published_at": safe_str(row.get("min_video_published_at", "")),
        "max_video_published_at": safe_str(row.get("max_video_published_at", "")),
        "item_source_sheets": safe_str(row.get("item_source_sheets", "")),
        "video_titles": safe_str(row.get("sample_video_titles", "")),
        "video_descriptions_sample": safe_str(row.get("sample_video_descriptions", "")),
    }

    return (
        "Classify the following YouTube playlist metadata for the research topic:\n"
        "\"Social practices of public audit and corruption exposure in Ukraine.\"\n\n"
        "PLAYLIST METADATA JSON:\n"
        f"{json.dumps(data, ensure_ascii=False, indent=2)}"
    )


def parse_json_response(text: str) -> Dict[str, Any]:
    text = text.strip()

    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()

    return json.loads(text)


def call_model(client: OpenAI, row: pd.Series) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": build_user_prompt(row)},
        ],
        timeout=REQUEST_TIMEOUT,
    )

    content = response.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("Model returned empty response.")

    return parse_json_response(content)


def load_existing_results() -> pd.DataFrame:
    if not OUTPUT_XLSX.exists():
        return pd.DataFrame()

    try:
        return pd.read_excel(OUTPUT_XLSX, sheet_name="playlist_ai_analysis")
    except Exception:
        return pd.DataFrame()


def existing_keys(existing_df: pd.DataFrame) -> set:
    if existing_df.empty or "row_key" not in existing_df.columns:
        return set()
    return set(existing_df["row_key"].dropna().astype(str).tolist())


def list_to_pipe(value: Any) -> str:
    if isinstance(value, list):
        return " | ".join(str(x) for x in value)
    return safe_str(value)


def save_outputs(results: List[Dict[str, Any]], log_rows: List[Dict[str, Any]]) -> None:
    results_df = pd.DataFrame(results)
    log_df = pd.DataFrame(log_rows)

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        results_df.to_excel(writer, index=False, sheet_name="playlist_ai_analysis")

    log_df.to_excel(LOG_XLSX, index=False)


def main() -> None:
    api_key = load_api_key()
    client = OpenAI(api_key=api_key)

    df = prepare_data()

    existing_df = load_existing_results()
    processed_keys = existing_keys(existing_df) if SKIP_EXISTING else set()

    results: List[Dict[str, Any]] = []
    log_rows: List[Dict[str, Any]] = []

    if not existing_df.empty and SKIP_EXISTING:
        results.extend(existing_df.to_dict(orient="records"))

    print(f"Loaded {len(df)} playlist rows for AI analysis.")
    print(f"Already processed: {len(processed_keys)}")
    print(f"Output: {OUTPUT_XLSX}")

    for idx, row in df.iterrows():
        row_key = make_row_key(row)

        if not row_key:
            log_rows.append({
                "row_index": idx,
                "row_key": "",
                "channel_title": safe_str(row.get("channel_title", "")),
                "playlist_title": safe_str(row.get("playlist_title", "")),
                "status": "failed",
                "error": "Cannot construct row_key",
            })
            continue

        if SKIP_EXISTING and row_key in processed_keys:
            print(f"Skipping existing: {row_key}")
            continue

        print(
            f"Processing row {idx}: "
            f"{safe_str(row.get('channel_title', ''))} / "
            f"{safe_str(row.get('playlist_title', ''))}"
        )

        last_error: Optional[Exception] = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                ai = call_model(client, row)

                result_row = {
                    "row_key": row_key,
                    "source_row_index": idx,

                    "list_source_sheet": safe_str(row.get("list_source_sheet", "")),
                    "playlist_id": safe_str(row.get("playlist_id", "")),
                    "channel_id": safe_str(row.get("channel_id", "")),
                    "channel_title": safe_str(row.get("channel_title", "")),
                    "playlist_title": safe_str(row.get("playlist_title", "")),
                    "playlist_url": safe_str(row.get("playlist_url", "")),
                    "playlist_description": safe_str(row.get("description", "")),
                    "declared_item_count": safe_str(row.get("item_count", "")),
                    "actual_video_count_from_items": safe_str(row.get("actual_video_count_from_items", "")),
                    "all_video_titles_count": safe_str(row.get("all_video_titles_count", "")),
                    "min_video_published_at": safe_str(row.get("min_video_published_at", "")),
                    "max_video_published_at": safe_str(row.get("max_video_published_at", "")),
                    "item_source_sheets": safe_str(row.get("item_source_sheets", "")),

                    "playlist_topic": ai.get("playlist_topic", ""),
                    "secondary_topics": list_to_pipe(ai.get("secondary_topics", [])),
                    "playlist_function": ai.get("playlist_function", ""),
                    "playlist_type": ai.get("playlist_type", ""),
                    "corruption_relevance_category": ai.get("corruption_relevance_category", ""),
                    "public_audit_relevance_category": ai.get("public_audit_relevance_category", ""),
                    "ukraine_relevance_category": ai.get("ukraine_relevance_category", ""),
                    "playlist_relevance_score": ai.get("playlist_relevance_score", ""),
                    "classification_confidence": ai.get("classification_confidence", ""),
                    "inclusion_status": ai.get("inclusion_status", ""),
                    "manual_review_required": ai.get("manual_review_required", ""),
                    "evidence_markers": list_to_pipe(ai.get("evidence_markers", [])),
                    "risk_notes": list_to_pipe(ai.get("risk_notes", [])),
                    "short_rationale": ai.get("short_rationale", ""),
                }

                results.append(result_row)

                log_rows.append({
                    "row_index": idx,
                    "row_key": row_key,
                    "channel_title": safe_str(row.get("channel_title", "")),
                    "playlist_title": safe_str(row.get("playlist_title", "")),
                    "status": "ok",
                    "error": "",
                })

                last_error = None
                break

            except Exception as exc:
                last_error = exc
                wait_seconds = attempt * 3
                print(f"  Attempt {attempt} failed: {exc}")

                if attempt < MAX_RETRIES:
                    print(f"  Retrying in {wait_seconds} seconds...")
                    time.sleep(wait_seconds)

        if last_error is not None:
            log_rows.append({
                "row_index": idx,
                "row_key": row_key,
                "channel_title": safe_str(row.get("channel_title", "")),
                "playlist_title": safe_str(row.get("playlist_title", "")),
                "status": "failed",
                "error": str(last_error),
            })

        save_outputs(results, log_rows)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print("Done.")
    print(f"Results saved to: {OUTPUT_XLSX}")
    print(f"Log saved to: {LOG_XLSX}")
    print(f"Diagnostics saved to: {DIAGNOSTICS_XLSX}")


if __name__ == "__main__" and RUN_ANALYSIS:
    prepare_run(MODEL_NAME, OUTPUT_XLSX, LOG_XLSX)
    main()
elif __name__ == "__main__":
    print("Preview mode: review configuration, then set RUN_ANALYSIS = True to execute this stage.")